# INTRODUCTION TO GEN AI
## SESSION 2

Assignment questions and answers


## Question 1

Build a simple autoencoder in Python using Keras to compress and reconstruct 28x28 grayscale images from the Fashion MNIST dataset. Train for 5 epochs and display one original and its reconstructed image side by side.

**Answer:**

```python
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import Dense, Flatten, Input, Reshape
from tensorflow.keras.models import Model

# Load and normalize data
(x_train, _), (x_test, _) = fashion_mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Define Autoencoder Architecture
input_img = Input(shape=(28, 28))
x = Flatten()(input_img)
encoded = Dense(64, activation="relu")(x)
decoded = Dense(784, activation="sigmoid")(encoded)
output_img = Reshape((28, 28))(decoded)

autoencoder = Model(input_img, output_img)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

# Train the model
autoencoder.fit(
    x_train, x_train, epochs=5, batch_size=256, validation_data=(x_test, x_test)
)

# Reconstruct and display one image
reconstructed = autoencoder.predict(x_test[:1])

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.title("Original")
plt.imshow(x_test[0], cmap="gray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Reconstructed")
plt.imshow(reconstructed[0], cmap="gray")
plt.axis("off")

plt.show()
```

## Question 2

Modify your autoencoder's loss function to use Binary Crossentropy (BCE) instead of Mean Squared Error (MSE), then compare the reconstruction quality visually for 3 test images.

Hint: Use model.compile(loss='binary_crossentropy', optimizer='adam').

**Answer:**

```python
import matplotlib.pyplot as plt

# 1. Compile model with Binary Crossentropy
autoencoder.compile(loss="binary_crossentropy", optimizer="adam")
autoencoder.fit(
    x_train,
    x_train,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test),
)

# 2. Reconstruct 3 test images
reconstructed = autoencoder.predict(x_test[:3])

# 3. Visually compare original vs reconstructed
plt.figure(figsize=(6, 4))
for i in range(3):
    # Original
    ax = plt.subplot(2, 3, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Reconstructed
    ax = plt.subplot(2, 3, i + 4)
    plt.imshow(reconstructed[i].reshape(28, 28), cmap="gray")
    plt.title("BCE Recon")
    plt.axis("off")

plt.tight_layout()
plt.show()
```

## Question 3

Use your trained autoencoder to denoise images: add random noise to 10 Fashion MNIST test images, pass them through the autoencoder, and display the noisy and denoised outputs together.

**Answer:**

```python
import matplotlib.pyplot as plt
import numpy as np

# Select 10 test images (assumes test_images is normalized between 0 and 1)
sample_images = test_images[:10]

# Add random Gaussian noise and clip to [0, 1]
noise_factor = 0.2
noisy_images = sample_images + noise_factor * np.random.normal(
    loc=0.0, scale=1.0, size=sample_images.shape
)
noisy_images = np.clip(noisy_images, 0.0, 1.0)

# Denoise using the trained autoencoder
denoised_images = autoencoder.predict(noisy_images)

# Display noisy and denoised images
plt.figure(figsize=(20, 4))
for i in range(10):
    # Noisy Image
    ax = plt.subplot(2, 10, i + 1)
    plt.imshow(noisy_images[i].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    # Denoised Output
    ax = plt.subplot(2, 10, i + 1 + 10)
    plt.imshow(denoised_images[i].squeeze(), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")

plt.show()
```

## Question 4

Build a Variational Autoencoder (VAE) using Keras for the Fashion MNIST dataset. After training, sample 5 random points from the latent space and generate new images from these points.

Hint: Use the VAE's decoder to generate images from sampled latent vectors.

**Answer:**

```python
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


# 1. Sampling layer
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


# 2. VAE Model class
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.losses.binary_crossentropy(data, reconstruction),
                    axis=(1, 2),
                )
            )
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1,
                )
            )
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"loss": total_loss}


# 3. Load and preprocess data
(x_train, _), _ = tf.keras.datasets.fashion_mnist.load_data()
x_train = np.expand_dims(x_train, -1).astype("float32") / 255.0

# 4. Build Encoder
latent_dim = 2
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(128, activation="relu")(x)
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)
z = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# 5. Build Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation="relu")(latent_inputs)
x = layers.Dense(28 * 28, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28, 1))(x)
decoder = models.Model(latent_inputs, decoder_outputs, name="decoder")

# 6. Train VAE
vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")
vae.fit(x_train, epochs=5, batch_size=128)

# 7. Sample 5 points from latent space and generate images
random_latent_points = np.random.normal(size=(5, latent_dim))
generated_images = decoder.predict(random_latent_points)

# 8. Display generated images
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i in range(5):
    axes[i].imshow(generated_images[i].squeeze(), cmap="gray")
    axes[i].axis("off")
plt.show()
```

## Question 5

Use ChatGPT to explain in your own words what KL-divergence loss does in a VAE and why it is needed. Paste your prompt and ChatGPT's response as your answer.

**Answer:**

**Prompt:**
Explain in simple terms what KL-divergence loss does in a Variational Autoencoder (VAE) and why it is needed.

**ChatGPT's Response:**
In a VAE, the KL-divergence loss measures how much the learned latent distribution differs from a standard normal distribution (a bell curve centered at zero). 

It acts as a regularizer. Without it, the encoder might scatter data points far apart in the latent space just to make reconstruction easier, creating gaps where the decoder has no idea what to do. By forcing the latent space to follow a standard normal distribution, KL-divergence keeps the space continuous, smooth, and compact. This ensures that when you randomly sample points to generate new data, the outputs are meaningful rather than corrupted noise.